# Python Core Patterns

A hands-on notebook covering four fundamental Python patterns:
- **Comprehensions** — list and dictionary, from simple to nested
- **Functional tools** — `map()`, `filter()`, and `lambda` for clean transformations
- **Text processing** — tokenization, cleaning, and vocabulary analysis on a real speech
- **OOP** — a generic n-dimensional `Vector` class with distance and arithmetic

Each section includes the reasoning behind the approach, not just the code.

---
## 1. Comprehensions

### 1a. Squaring a list of numbers

List comprehensions are the idiomatic Python way to apply a transformation to every element — cleaner than a `for` loop with `.append()` and more readable than `map()` for simple expressions.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
squares = [n * n for n in numbers]

print("Original:", numbers)
print("Squares: ", squares)

### 1b. Title-casing strings

Title casing capitalizes the first letter of every word — `"green apple"` → `"Green Apple"`. Python's built-in `.title()` handles multi-word strings correctly, making it a natural fit inside a comprehension.

In [ ]:
fruits = ['green apple', 'banana', 'cherry', 'date']
fruits_title = [fruit.title() for fruit in fruits]

print("Original:    ", fruits)
print("Title-cased: ", fruits_title)

### 1c. Nested dictionary comprehension

Building a nested structure where outer keys are integers 1–3 and inner keys are letters `'a'`, `'b'`, `'c'`, with values equal to `outer × letter_position`.

The trick: `ord('a') = 97`, so `ord(ch) - 96` gives the letter's alphabet position (a=1, b=2, c=3).

I first worked through it as a regular nested loop to verify the logic, then collapsed it into a one-liner.

In [ ]:
# Step 1: understand with a regular loop
nested_dictionary = {}
for i in range(1, 4):
    inner = {}
    for ch in ['a', 'b', 'c']:
        inner[ch] = i * (ord(ch) - 96)
    nested_dictionary[i] = inner

print("Loop version:", nested_dictionary)

In [ ]:
# Step 2: collapse into a nested dictionary comprehension
nested_dictionary = {
    i: {ch: i * (ord(ch) - 96) for ch in ['a', 'b', 'c']}
    for i in range(1, 4)
}

print("Comprehension:", nested_dictionary)

---
## 2. Functional Tools: `map()`, `filter()`, `lambda`

### 2a. Binary strings → decimal integers

Python's `int(x, base)` converts a string from any base to decimal. `int('1010', 2)` reads `'1010'` as binary and returns `10`. Wrapping this in `map()` with a lambda applies it across the whole list in one pass.

In [ ]:
binaries = ['1010', '1111', '0001', '100000']
decimals = list(map(lambda b: int(b, 2), binaries))

print("Binary: ", binaries)
print("Decimal:", decimals)

### 2b. Filtering palindromes

A palindrome reads the same forwards and backwards. Python's slice `word[::-1]` reverses a string in one expression — pairing it with `filter()` keeps only the matching words.

In [ ]:
words = ['level', 'world', 'deified', 'python', 'radar']
palindromes = list(filter(lambda w: w == w[::-1], words))

print("All words:   ", words)
print("Palindromes: ", palindromes)

### 2c. Squares of even numbers — one-liner

Chaining `filter()` and `map()`: first keep only even numbers, then square each. Reading inside-out: filter runs first, map runs on its output.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
even_squares = list(map(lambda x: x * x, filter(lambda x: x % 2 == 0, numbers)))

print("Even squares:", even_squares)

### 2d. Filtering prime numbers

A number is prime if it's greater than 1 and has no divisors other than 1 and itself. Key optimization: only check divisors up to √n — any factor larger than √n must be paired with one smaller than √n, so we'd have already found it. Also skip even numbers after checking 2, halving the iterations.

In [ ]:
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, int(n**0.5) + 1, 2):  # step 2: skip even divisors
        if n % d == 0:
            return False
    return True

ints = list(range(1, 21))
prime_numbers = list(filter(is_prime, ints))

print("Input:  ", ints)
print("Primes: ", prime_numbers)

---
## 3. Text Processing: Gettysburg Address

A mini NLP pipeline on a real historical text — tokenization, cleaning, and vocabulary analysis.

### 3a. Load the text

In [ ]:
with open('gettysburg.txt', 'r', encoding='utf-8') as f:
    contents = f.read()

print(f"Loaded {len(contents)} characters")
print("\nFirst 200 chars:")
print(contents[:200])

### 3b. Tokenize into words

`.split()` splits on any whitespace (spaces, newlines, tabs) and discards them — giving a raw list of tokens. At this stage punctuation is still attached to words (e.g. `'nation,'`).

In [ ]:
def speech_to_list(text):
    """Split text into a flat list of whitespace-delimited tokens."""
    speech_list = []
    for line in text.splitlines():
        speech_list.extend(line.split())
    return speech_list

speech_list = speech_to_list(contents)
print(f"Total tokens: {len(speech_list)}")
print("First 15:", speech_list[:15])

### 3c. Remove non-words

The raw token list includes `'--'` (Lincoln's em-dash marker) which isn't a word. Filtering it out with a one-by-one check using `.append()` gives us more control than a list comprehension — easy to extend with additional exclusion rules later.

In [ ]:
def speech_to_list_better(token_list):
    """Filter out non-word tokens like '--'."""
    result = []
    for word in token_list:
        if word != '--':
            result.append(word)
    return result

speech_list_better = speech_to_list_better(speech_list)
print(f"After removing '--': {len(speech_list_better)} tokens (was {len(speech_list)})")

### 3d. Unique vocabulary

Two normalization rules before deduplication:
- **Case:** `"We"` and `"we"` are the same word → `.lower()`
- **Punctuation:** `"here,"` `"here."` `"here"` are the same word → `.strip(string.punctuation)`

Using a `set` for O(1) deduplication, then sorting for deterministic output.

In [ ]:
import string

def unique_words(word_list):
    """Return sorted list of unique words, case- and punctuation-normalized."""
    unique = set()
    for word in word_list:
        normalized = word.lower().strip(string.punctuation)
        if normalized and normalized != '--':
            unique.add(normalized)
    return sorted(unique)

speech_list_unique = unique_words(speech_list_better)
print(f"Unique vocabulary: {len(speech_list_unique)} words")
print("\nSummary:")
print(f"  Raw tokens      : {len(speech_list)}")
print(f"  After cleaning  : {len(speech_list_better)}")
print(f"  Unique words    : {len(speech_list_unique)}")
print("\nFirst 20 unique words:", speech_list_unique[:20])

---
## 4. Object-Oriented Design: n-Dimensional Vector

A generic `Vector` class that works in any number of dimensions, with Euclidean distance, vector addition, and a readable string representation.

### Design notes

- `__init__`: stores `dimension` and `coordinates`; defaults to 2D zero vector
- `distance()`: Euclidean distance — √Σ(xᵢ - yᵢ)²
- `vector_sum()`: returns a new `Vector` — immutable pattern, no mutation
- `__str__()`: Python calls this when you `print()` an object; without it you'd see `<__main__.Vector object at 0x...>` instead of something readable
- `norm()`: 2-norm (vector length) — √Σxᵢ² — equivalent to distance from origin

In [ ]:
import math

class Vector:
    """A generic n-dimensional vector with distance, addition, and norm."""

    def __init__(self, dimension=2, coordinates=None):
        self.dimension = dimension
        self.coordinates = coordinates if coordinates is not None else [0.0] * dimension

    def distance(self, other):
        """Euclidean distance between self and other."""
        return math.sqrt(sum((a - b) ** 2 for a, b in zip(self.coordinates, other.coordinates)))

    def vector_sum(self, other):
        """Return a new Vector whose coordinates are the element-wise sum."""
        summed = [a + b for a, b in zip(self.coordinates, other.coordinates)]
        return Vector(self.dimension, summed)

    def norm(self):
        """2-norm (length) of the vector — equivalent to distance from origin."""
        return math.sqrt(sum(c ** 2 for c in self.coordinates))

    def __str__(self):
        return f"Vector in {self.dimension}D: {self.coordinates}"

### Worked example: 5D vector

Given **x = [1.0, -1.0, 3.0, 5.0, -2.2]** and **y = origin (5D zero vector)**:
- Distance from x to y = √(1 + 1 + 9 + 25 + 4.84) = √40.84 ≈ 6.3906
- z = x + y = x (adding zero vector changes nothing)
- ‖z‖ = ‖x‖ ≈ 6.3906
- `print(z)` calls `__str__()` — no direct attribute access needed

In [ ]:
x = Vector(5, [1.0, -1.0, 3.0, 5.0, -2.2])
y = Vector(5, [0.0, 0.0, 0.0, 0.0, 0.0])   # 5D origin

print("x:", x)
print("y:", y)

# Distance from x to origin
dist = x.distance(y)
print(f"\nDistance x → origin : {dist:.4f}")

# z = x + y
z = x.vector_sum(y)

# Length (2-norm) of z
print(f"‖z‖ (2-norm)        : {z.norm():.4f}")

# Print z without accessing its attributes directly — __str__() handles it
print("\nz coordinates (via __str__):", z)

### Quick sanity check: 2D case

Classic 3-4-5 right triangle: distance from [3, 4] to origin should be 5.

In [ ]:
v1 = Vector(2, [3.0, 4.0])
v2 = Vector(2, [0.0, 0.0])

print(f"Distance (expect 5.0): {v1.distance(v2)}")
print(f"Sum: {v1.vector_sum(v2)}")